# Day 4: Social Science Applications
## Information Extraction, Summarization, and RAG

**LLMs for Social Science** | Oxford Spring School

| Day | Theme | Status |
|-----|-------|--------|
| 1 | From Embeddings to Transformers | Done |
| 2 | From Models to Tools | Done |
| 3 | Deploying for Research | Done |
| **4** | **Social Science Applications** | **Today** |
| 5 | Agentic Workflows | Next |

---

A colleague asks you:

> *"What does the recent political science literature say about how states are approaching AI governance, and how does this connect to democratic backsliding?"*

You have ~100 review articles from the *Annual Review of Political Science* (2021–2024). You could read all of them. That would take days. Or you could build a system that reads them for you — extracting structured claims, retrieving relevant passages, and generating grounded answers.

By the end of today, you'll have built exactly that system. Along the way, you'll learn:

1. **Information Extraction** — How to get LLMs to pull structured data (claims, findings, methods) from academic text, and how to verify they're not making things up
2. **Retrieval-Augmented Generation (RAG)** — How to combine the embeddings from Day 1 with the prompting from Day 2 to answer questions over large corpora
3. **When to trust the output** — The thread running through everything: provenance, faithfulness, and evaluation

**Prerequisites**: Days 1–3 of this course. You'll use embeddings (Day 1), prompting (Day 2), and the local model from Day 3.

**Today's dataset**: ~100 review articles from the *Annual Review of Political Science* (2021–2024), covering topics from democratic backsliding to AI governance to immigration policy.

## Setup

### Installing packages

We need a few new libraries today:

- **`PyMuPDF`** (`fitz`): Extracts text from PDF files. It reads each page and returns the text content, handling the PDF format's internal structure (fonts, positioning, etc.) so we don't have to.
- **`sentence-transformers`**: Provides pre-trained embedding models that convert text into dense vectors (the same kind of vectors from Day 1, but from a more powerful model). We'll use `all-MiniLM-L6-v2` — a small, fast model that produces 384-dimensional embeddings.
- **`faiss-cpu`**: Facebook AI Similarity Search. An efficient library for finding the most similar vectors in a large collection. Think of it as a database optimized for "find me the k closest vectors to this query vector" — exactly what we need for retrieval.

The transformers and torch packages carry over from previous days.

In [1]:
# Install packages (takes ~2 minutes)
!pip install -q PyMuPDF sentence-transformers faiss-cpu openai
!pip install -q transformers torch accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.7 MB/s eta 0:00:00


### Load the data

The cell below downloads four years of article PDFs (2021–2024) from the course GitHub repository and extracts text from each one. That's roughly 100 review articles from the *Annual Review of Political Science* — a serious corpus.

> **What the code does**: For each year, it downloads and unzips the PDFs. Then, for each PDF, `fitz` reads page by page, extracts the raw text, and strips out the download headers that appear on every page. We also parse the title from the first page's layout.

In [2]:
import fitz  # PyMuPDF
import os
import json
import re
import zipfile
import urllib.request

# ── Download articles from the course repository ──
BASE_URL = "https://github.com/antndlcrx/Intro-to-LLMs-DPIR/raw/main/data/articles"
YEARS = [2021, 2022, 2023, 2024]
ARTICLES_DIR = "articles"

os.makedirs(ARTICLES_DIR, exist_ok=True)

for year in YEARS:
    pdf_dir = os.path.join(ARTICLES_DIR, str(year))
    if not os.path.exists(pdf_dir):
        url = f"{BASE_URL}/{year}.zip"
        zip_path = os.path.join(ARTICLES_DIR, f"{year}.zip")
        print(f"Downloading {year}.zip...")
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(ARTICLES_DIR)
        print(f"  → {len(os.listdir(pdf_dir))} PDFs extracted")
    else:
        print(f"{year}: already downloaded ({len(os.listdir(pdf_dir))} PDFs)")


def extract_articles(articles_dir, years):
    """Extract text and metadata from all PDFs across multiple year directories.

    Returns a dict mapping article_id -> {title, filename, text, pages, year}.
    Each article's text is cleaned of download headers and page footers.
    """
    articles = {}

    for year in years:
        pdf_dir = os.path.join(articles_dir, str(year))
        if not os.path.exists(pdf_dir):
            continue

        for filename in sorted(os.listdir(pdf_dir)):
            if not filename.endswith('.pdf'):
                continue

            doc = fitz.open(os.path.join(pdf_dir, filename))

            # Extract and clean text from all pages
            pages_text = []
            for page in doc:
                text = page.get_text()
                # Remove the download header that Annual Reviews adds
                lines = [l for l in text.split('\n')
                         if not l.startswith('Downloaded from www.annualreviews.org')]
                pages_text.append('\n'.join(lines))

            full_text = '\n'.join(pages_text)

            # Parse title from first page
            first_page_lines = [l.strip() for l in pages_text[0].split('\n') if l.strip()]
            title_lines = []
            capture = False
            for line in first_page_lines:
                if 'Annual Review of Political Science' in line:
                    capture = True
                    continue
                if capture:
                    if any(x in line for x in ['email:', 'Department of', 'Annu. Rev.',
                                                'First published', 'Copyright', 'licensed']):
                        if title_lines:
                            break
                        continue
                    if line and len(line) > 2:
                        title_lines.append(line)
                        if len(title_lines) >= 3:
                            break

            article_id = filename.replace('.pdf', '')
            articles[article_id] = {
                'title': ' '.join(title_lines).strip(),
                'filename': filename,
                'pdf_dir': pdf_dir,
                'text': full_text,
                'pages': len(doc),
                'year': year,
            }
            doc.close()

    return articles


articles = extract_articles(ARTICLES_DIR, YEARS)
print(f"\nLoaded {len(articles)} articles across {len(YEARS)} years\n")
for year in YEARS:
    year_articles = [a for a in articles.values() if a['year'] == year]
    print(f"  {year}: {len(year_articles)} articles")

  → 22 PDFs extracted
  → 26 PDFs extracted
  → 25 PDFs extracted
  → 24 PDFs extracted

Loaded 97 articles across 4 years

  2021: 22 articles
  2022: 26 articles
  2023: 25 articles
  2024: 24 articles


### Load the local model

Same model as Days 2–3: **Qwen2.5-3B-Instruct**. A small model that runs on a free Colab T4 GPU. We load it in 4-bit quantization to fit in memory.

We also define a helper function `generate()` that wraps the chat template and generation call — you'll use this throughout the notebook.

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# 4-bit quantization so it fits on a T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
print(f"Loaded {MODEL_NAME} on {model.device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-3B-Instruct on cpu


In [5]:
def generate(prompt, system="You are a helpful research assistant.", max_new_tokens=1024, temperature=0.1):
    """Send a prompt to the local model and return the response text.

    Args:
        prompt: The user message.
        system: System message that sets the model's role.
        max_new_tokens: Maximum length of the response.
        temperature: Controls randomness (0.0 = deterministic, 1.0 = creative).

    Returns:
        The model's response as a string.
    """
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the new tokens (skip the prompt)
    response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

# Quick test
print(generate("What is information extraction in NLP? Answer in one sentence."))

KeyboardInterrupt: 

---

# Section 1: Information Extraction

You've classified text (Day 3). Now you'll do something harder: **extract structured information** from unstructured academic prose. This is the difference between asking "is this text about democracy?" (classification) and asking "what specific claims does this text make about democracy, and what evidence does it cite?" (extraction).

This matters for social science because review articles — like the ones in our corpus — are dense with claims, cited findings, and methodological discussions. Manually cataloguing all of this across ~100 articles would take weeks. An LLM can attempt it in seconds. The question is: **can you trust what it extracts?**

Before we ask the model to extract anything, you'll do it yourself. This is the "Be the Algorithm" exercise: you need to understand what good extraction looks like before you can evaluate whether a model is doing it well.

## Exercise 1: Be the Algorithm — Manual Extraction

Below is a passage from one of the review articles. Read it carefully, then extract the following into a structured format:

1. **Claims**: Substantive assertions the authors make (e.g., "X leads to Y", "there is evidence that Z")
2. **Cited findings**: Specific results attributed to other researchers (e.g., "Smith (2020) found that...")
3. **Methodological approaches**: Any methods, frameworks, or analytical approaches mentioned

This is codebook design — the same skill from Day 3's classification, applied to a harder task. You're deciding *what counts* as a claim vs. a cited finding vs. a method. There's no single right answer, but there are clearly wrong ones (missing obvious claims, or inventing things that aren't in the text).

Think about the categories — do they cleanly separate? Is a "cited finding" also a "claim"? These ambiguities are real and matter when you automate this.

In [6]:
# Let's pick a substantive passage from the corpus
# We'll use the AI governance article — it's rich with claims and citations

ai_article = [a for a in articles.values() if 'Governance of Artificial' in a['title']][0]

# Extract a meaty passage from the body (pages 3-5, where the argument develops)
doc = fitz.open(os.path.join(ai_article['pdf_dir'], ai_article['filename']))
passage_pages = []
for pg_num in [2, 3]:  # 0-indexed; these are pages 3-4
    text = doc[pg_num].get_text()
    lines = [l for l in text.split('\n') if not l.startswith('Downloaded')]
    passage_pages.append('\n'.join(lines))
doc.close()

passage = '\n'.join(passage_pages)

# Trim to a manageable chunk (~1500 chars) — find a good paragraph boundary
# We want the section on how AI governance differs from previous challenges
start_marker = "HOW AI GOVERNANCE DIFFERS"
end_idx = passage.find("STATE APPROACHES")
if start_marker in passage and end_idx > 0:
    passage = passage[passage.find(start_marker):end_idx].strip()
elif len(passage) > 3000:
    passage = passage[500:2500]  # fallback: take middle section

print("=" * 80)
print(f"PASSAGE FROM: {ai_article['title']}")
print("=" * 80)
print(passage[:3000])
if len(passage) > 3000:
    print(f"\n... [{len(passage) - 3000} more characters]")

PASSAGE FROM: Terra Incognita: The Governance of Artificial Intelligence in Global
opportunities for collaboration, which provide additional insurance against the negative
consequences of the accelerating velocity of change humanity now faces.
HOW AI GOVERNANCE DIFFERS FROM PREVIOUS
GOVERNANCE CHALLENGES
The modern world is full of complex machines that users rely on experts to understand; if my car
breaks down, I do not know how to fix it, but a mechanic probably can. Generative AI is something
else entirely. No single human understands precisely how it arrives at its responses, at least not
in the sense of an Enlightenment-style cause-and-effect sequence. If it goes off the rails, as it
notably did when it tried to persuade New York Times reporter Kevin Roose that he should divorce
his wife and instead devote himself to his new silicon friend, a band-aid such as reinforcement
learning from human feedback (RLHF) can be applied to the model to prevent similar behavior
in the future. RL

### Your turn

In the cell below, manually extract claims, cited findings, and methods from the passage above. Use the dictionary structure provided. Be specific — quote or closely paraphrase the text. Include the citation where the text provides one.

**Why this matters**: You're building a gold standard. When the model does this extraction next, you'll compare its output against yours. If you rush this, you won't be able to evaluate the model properly.

In [8]:
# @title Exercise 1: Manual extraction
# Read the passage above and fill in the lists below (3-5 items per category).

manual_extraction = {
    "claims": [
        # Example: "AI governance differs from previous governance challenges because..."
        # TODO: Add 3-5 claims from the passage
        "No human understands prexisely how AI generates responses",
        "RLHF does not solve the AI governance problem entirely",
        ""
    ],
    "cited_findings": [
        {"finding": "RLHF relies on human ghost work, the invisible behind-the-scenes labor whereby human grooming of raw data makes generative AI possible", "source": "Gray & Suri (2019)"},
        {"finding": """Additionally, there are limitations to the safety benefits of RLHF, and with the advent of open-
source foundation models that can run on a laptop, some future users might intentionally hijack
ostensibly benign models and turn them to malevolent purposes""", "source": "Casper et al. (2023)"},
        {"finding": "only wealthy companies can muster the resources to bring SOTA foundation models into being", "source": "Knight (2023), Maslej et al. (2023)"}
        # TODO: Add 3-5 cited findings
    ],
    "methods_or_frameworks": [
        # Example: "The authors use a comparative framework across open and closed societies"
        # TODO: Add any methods/frameworks mentioned
        "literature review",
        "case studies"
    ],
}

# Print your extraction for review
import json
print(json.dumps(manual_extraction, indent=2))
print(f"\nYou extracted: {len(manual_extraction['claims'])} claims, "
      f"{len(manual_extraction['cited_findings'])} cited findings, "
      f"{len(manual_extraction['methods_or_frameworks'])} methods")

{
  "claims": [
    "No human understands prexisely how AI generates responses",
    "RLHF does not solve the AI governance problem entirely",
    ""
  ],
  "cited_findings": [
    {
      "finding": "RLHF relies on human ghost work, the invisible behind-the-scenes labor whereby human grooming of raw data makes generative AI possible",
      "source": "Gray & Suri (2019)"
    },
    {
      "finding": "Additionally, there are limitations to the safety benefits of RLHF, and with the advent of open-\nsource foundation models that can run on a laptop, some future users might intentionally hijack\nostensibly benign models and turn them to malevolent purposes",
      "source": "Casper et al. (2023)"
    },
    {
      "finding": "only wealthy companies can muster the resources to bring SOTA foundation models into being",
      "source": "Knight (2023), Maslej et al. (2023)"
    }
  ],
  "methods_or_frameworks": [
    "literature review",
    "case studies"
  ]
}

You extracted: 3 claims, 3 

## Exercise 2: Automated Extraction — Write the Prompt

Now write a prompt that instructs the model to perform the same extraction you just did manually. This is where the Day 2 prompting skills pay off.

Key design decisions you'll face:
- **How specific should the schema be?** Should you define what a "claim" is, or let the model decide?
- **Should you ask for JSON output?** (Hint: yes — you need to parse the results programmatically)
- **Should you include an example?** (Few-shot vs. zero-shot — you experimented with this on Day 2)
- **How do you enforce provenance?** Each extracted item should point back to the source text.

Remember from Day 2: the prompt *is* your instrument. Small changes in wording can change what gets extracted.

In [11]:
# @title Exercise 2: Write an extraction prompt
# Hints:
# - Define what each category means
# - Ask for a source_quote for each item (provenance!)
# - Specify the JSON structure you want
# - Tell the model what NOT to do (don't invent, don't add info not in the passage)

extraction_prompt = f"""
# TODO: Write your extraction prompt here.
# The variable `passage` contains the text to extract from.
# Your prompt should ask for JSON output with claims, cited_findings,
# and methods_or_frameworks.

Please extract the following information from the passage in JSON format:
- `claims`: 3-5 claims. Example for one: AI governance differs from previous governance challenges because...
- `cited_findings`: 3-5 cited findings. Example for one: {{"finding": "...", "source": "Author (Year)"}}
- `methods_or_frameworks`: The authors use a comparative framework across open and closed societies.

Passage:
{passage}
"""

# Run extraction with the local model
llm_response = generate(extraction_prompt, max_new_tokens=1500, temperature=0.1)
print(llm_response)

KeyboardInterrupt: 

In [ ]:
# Parse the JSON from the model's response
import re

def parse_json_response(response_text):
    """Extract JSON from a model response that may contain markdown formatting.

    Models often wrap JSON in ```json ... ``` blocks. This function handles
    that, plus responses where the JSON is embedded in surrounding text.
    Returns the parsed dict, or None if parsing fails.
    """
    # Try to find JSON in code blocks first
    json_match = re.search(r'```(?:json)?\s*\n?(.*?)```', response_text, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(1))
        except json.JSONDecodeError:
            pass

    # Try parsing the whole response as JSON
    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        pass

    # Try finding a JSON object in the text
    json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass

    return None


llm_extraction = parse_json_response(llm_response)
if llm_extraction:
    print(json.dumps(llm_extraction, indent=2))
else:
    print("⚠️  Could not parse JSON from model response.")
    print("Tip: Adjust your prompt to be more specific about the output format.")
    print("\nRaw response:")
    print(llm_response[:500])

## Faithfulness Verification

You now have two extractions: yours (manual) and the model's (automated). But the critical question isn't just "do they agree?" — it's **"is the model's extraction faithful to the source text?"**

Faithfulness means: every claim the model extracted should be traceable to a specific passage in the source text. If the model says the article claims X, you should be able to find where it says X.

This is the core methodological concern for social scientists using LLMs for extraction. A model that invents plausible-sounding claims is worse than useless — it contaminates your data.

Below is a simple verification function. For each extracted claim, it checks two things: (1) do key terms from the claim appear in the source text? (2) if the model provided a source quote, does that quote actually exist in the passage? Run it on your extraction results and study the output.

In [ ]:
def verify_faithfulness(extracted_items, source_text):
    """Check whether extracted claims are supported by the source text.

    For each claim, checks keyword overlap and whether any provided
    source_quote actually appears in the passage. Returns a verdict:
    'supported', 'partial', or 'unsupported'.
    """
    source_lower = source_text.lower()
    results = []

    for item in extracted_items:
        claim = item if isinstance(item, str) else item.get('claim', item.get('finding', str(item)))
        source_quote = item.get('source_quote', '') if isinstance(item, dict) else ''

        # Check 1: do key terms from the claim appear in the source?
        words = [w.lower().strip('.,;:()') for w in claim.split() if len(w) > 4]
        matches = sum(1 for w in words if w in source_lower)
        keyword_score = matches / max(len(words), 1)

        # Check 2: does the source_quote actually appear in the passage?
        quote_found = source_quote.lower()[:50] in source_lower if source_quote else False

        # Verdict
        if quote_found and keyword_score > 0.3:
            verdict = "supported"
        elif keyword_score > 0.4 or quote_found:
            verdict = "partial"
        else:
            verdict = "unsupported"

        results.append({
            "claim": claim[:120],
            "verdict": verdict,
            "keyword_overlap": f"{keyword_score:.0%}",
            "quote_in_source": quote_found,
        })

    return results


# Run verification on the model's extraction
if llm_extraction:
    # Adjust the key name if your extraction schema uses something different
    claims = llm_extraction.get("claims", [])
    verification = verify_faithfulness(claims, passage)

    supported = sum(1 for v in verification if v['verdict'] == 'supported')
    partial = sum(1 for v in verification if v['verdict'] == 'partial')
    unsupported = sum(1 for v in verification if v['verdict'] == 'unsupported')

    print(f"=== FAITHFULNESS VERIFICATION ({len(verification)} claims) ===")
    print(f"    Supported: {supported} | Partial: {partial} | Unsupported: {unsupported}\n")

    for v in verification:
        icon = {"supported": "✓", "partial": "~", "unsupported": "✗"}[v['verdict']]
        print(f"  {icon} {v['verdict']:12s} | keywords: {v['keyword_overlap']} | quote found: {v['quote_in_source']}")
        print(f"    {v['claim']}")
        print()
else:
    print("No parsed extraction available. Go back and fix Exercise 2.")

### Reflection

Before moving on, consider:

- **What kinds of errors did the model make?** Did it hallucinate claims, miss important ones, or extract things that were partially correct?
- **How would these errors affect a research pipeline?** If you used this extraction at scale across the full corpus, what would your dataset look like?
- **How does the extraction schema affect results?** Would you get different (better? worse?) extractions with a different schema definition?

> **Stretch goal**: Take a passage from a *different* article in the corpus. Run your extraction prompt on it. Does the prompt generalize, or was it overfit to the AI governance article's style? Try at least one article from a different subfield (e.g., the sanctions article, or the education politics article).

---

# Section 2: Retrieval-Augmented Generation (RAG)

In Section 1, you extracted information from a passage you *already had*. But what if you have a question and don't know *which* passages to look at? With ~100 articles averaging 20 pages each, that's roughly 2,000 pages of text. No model's context window can hold all of that, and even if it could, performance degrades with very long contexts.

**RAG solves this**: instead of feeding the model everything, you first *retrieve* the most relevant passages, then feed only those to the model for *generation*.

The retrieval step uses **embeddings** — the same vector representations from Day 1. You embed your question, embed all your passages, and find the passages whose vectors are closest to your question's vector. Then you pass those passages to the model as context.

The pipeline:
```
Question → Embed → Find similar chunks → Pass to LLM → Answer (with sources)
```

Before we build this, you'll first do it manually — to build intuition for what "relevance" means and how well embeddings capture it.

## Setting up the API model

In Section 1, we used our local 3B-parameter model. For RAG, generation quality matters more — the model needs to synthesize information from retrieved passages into a coherent answer *without* adding information that isn't there. Larger models are substantially better at this.

We'll use **Qwen3-235B-A22B** via the Nebius API. This is a Mixture-of-Experts model: 235 billion total parameters, but only 22 billion are active for any given input (the model routes each token to the most relevant 'expert' subnetworks). Compared to our local 3B model, that's a massive jump in capacity.

**To get an API key**: Register at [tokenfactory.nebius.com](https://tokenfactory.nebius.com). You get $1 of free credit, which is more than enough for this entire notebook.

> **What's happening architecturally**: Instead of running the model on your Colab GPU, you're sending requests over the internet to Nebius's servers, which run the model on much more powerful hardware. This is the same OpenAI-compatible API pattern from Day 3 — the `openai` Python library works with any provider that implements this standard.

In [7]:
from openai import OpenAI
from google.colab import userdata

# ── Set your API key ──
# Option 1: Use Colab secrets (recommended)
# Go to the key icon in the left sidebar → add NEBIUS_API_KEY
# Option 2: Paste directly (not recommended for shared notebooks)

try:
    API_KEY = userdata.get('API_KEY')
except Exception:
    API_KEY = "YOUR_KEY_HERE"  # ← replace with your key if not using secrets

api_client = OpenAI(
    base_url="https://api.deepseek.com/v1",
    api_key=API_KEY,
)

API_MODEL = "deepseek-chat"

def generate_api(prompt, system="You are a helpful research assistant.", max_tokens=1024, temperature=0.1):
    """Send a prompt to the API model and return the response text.

    Same interface as generate() but uses the API instead of the local model.
    """
    response = api_client.chat.completions.create(
        model=API_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return response.choices[0].message.content.strip()

# Test the API
print(f"Testing {API_MODEL}...")
test_response = generate_api("What is RAG in NLP? Answer in one sentence.")
print(f"Response: {test_response}")

Testing deepseek-chat...
Response: RAG (Retrieval-Augmented Generation) in NLP is a hybrid framework that enhances large language models by retrieving relevant information from external knowledge sources before generating responses, thereby improving accuracy and reducing hallucinations.


### Load the embedding model

We'll use **`all-MiniLM-L6-v2`** from `sentence-transformers`. This is a small (80MB) model that converts text into 384-dimensional vectors. It runs on CPU and is fast — embedding our entire corpus takes under a minute.

This is a *different kind* of model from the language models we've been using. It doesn't generate text — it only produces vectors. It was trained specifically so that semantically similar texts end up with similar vectors (high cosine similarity).

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model — runs on CPU, very fast
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Embedding model loaded: {embedder.get_sentence_embedding_dimension()}-dimensional vectors")

# Quick test: embed two sentences and check similarity
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

v1 = embedder.encode("Democratic backsliding weakens democratic institutions")
v2 = embedder.encode("Authoritarianism undermines democratic norms")
v3 = embedder.encode("The price of tomatoes has risen sharply")

print(f"\nSimilarity (backsliding ↔ authoritarianism): {cosine_similarity(v1, v2):.3f}")
print(f"Similarity (backsliding ↔ tomatoes):          {cosine_similarity(v1, v3):.3f}")

## Exercise 4: Be the Algorithm — Manual Relevance Ranking

Before we build the RAG pipeline, you'll do retrieval manually. This builds intuition for what embedding-based retrieval actually captures — and where it fails.

Below, you'll see a research question and 5 passages from different articles. Your task: **rank them by relevance** to the question, from most to least relevant. Then we'll see whether the embedding model agrees with you.

In [ ]:
# Pick 5 passages from different articles, varying in relevance to a question
research_question = "How do states approach the governance and regulation of artificial intelligence?"

# Select articles and extract representative passages
sample_articles = {
    'ai_governance': 'Governance of Artificial',
    'backsliding': 'Theories of Democratic Backsliding',
    'sanctions': 'Global Economic Sanctions',
    'corruption': 'Corruption and Social Norms',
    'state_capacity': 'Endogenous State Capacity',
}

passages_for_ranking = {}

for label, title_fragment in sample_articles.items():
    # Find the matching article
    art = [a for a in articles.values() if title_fragment in a['title']]
    if not art:
        continue
    art = art[0]

    # Get a passage from the body (page 3 or 4, skipping frontmatter)
    doc = fitz.open(os.path.join(art['pdf_dir'], art['filename']))
    for pg_num in [3, 4]:
        text = doc[pg_num].get_text()
        lines = [l for l in text.split('\n')
                 if not l.startswith('Downloaded') and len(l.strip()) > 0]
        clean = ' '.join(lines)
        if len(clean) > 400:
            # Take ~500 chars ending at a sentence boundary
            chunk = clean[:600]
            last_period = chunk.rfind('.')
            if last_period > 200:
                chunk = chunk[:last_period + 1]
            passages_for_ranking[label] = {
                'title': art['title'],
                'text': chunk
            }
            break
    doc.close()

# Display passages
print(f"RESEARCH QUESTION: {research_question}\n")
print("=" * 80)
for i, (label, p) in enumerate(passages_for_ranking.items()):
    print(f"\nPASSAGE {chr(65+i)} (from: {p['title'][:60]})")
    print("-" * 40)
    print(p['text'][:500])
    print()

In [ ]:
# @title Exercise 4: Rank passages by relevance
# Put the labels in order from MOST to LEAST relevant. Then we compare to embeddings.

# Replace with your ranking (list the labels in order, most relevant first)
your_ranking = [
    # TODO: e.g., "ai_governance", "state_capacity", "backsliding", "sanctions", "corruption"
]

# Now let's see what the embeddings say
question_vec = embedder.encode(research_question)

embedding_scores = {}
for label, p in passages_for_ranking.items():
    passage_vec = embedder.encode(p['text'])
    score = cosine_similarity(question_vec, passage_vec)
    embedding_scores[label] = score

# Sort by similarity (highest first)
embedding_ranking = sorted(embedding_scores.keys(), key=lambda x: embedding_scores[x], reverse=True)

print("YOUR RANKING vs EMBEDDING RANKING\n")
print(f"{'Rank':<6} {'Your pick':<20} {'Embedding pick':<20} {'Similarity':<10}")
print("-" * 56)
for i in range(len(embedding_ranking)):
    your_pick = your_ranking[i] if i < len(your_ranking) else "—"
    emb_pick = embedding_ranking[i]
    score = embedding_scores[emb_pick]
    match = "✓" if your_pick == emb_pick else ""
    print(f"{i+1:<6} {your_pick:<20} {emb_pick:<20} {score:.4f}  {match}")

print("\n💡 Where do you and the embedding model disagree? Why might that be?")

## Building the RAG Pipeline

Now we build the full pipeline. Three steps:

1. **Chunk**: Split each article into overlapping passages. We can't embed entire 20-page articles — we need smaller pieces that the model can use as context. Chunk size is a key design choice: too small and you lose context, too large and you dilute relevance.

2. **Embed**: Convert every chunk into a vector using our embedding model.

3. **Index**: Store all vectors in a FAISS index for fast similarity search. When a question comes in, we embed it and find the *k* most similar chunks.

In [ ]:
def chunk_text(text, chunk_size=500, overlap=100):
    """Split text into overlapping chunks of roughly chunk_size words.

    Overlap ensures that information spanning two chunks isn't lost.
    Each chunk includes the previous `overlap` words from the prior chunk.

    Args:
        text: The full article text.
        chunk_size: Target number of words per chunk.
        overlap: Number of words shared between consecutive chunks.

    Returns:
        List of text chunks.
    """
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap  # step forward by (chunk_size - overlap)
    return chunks


# Chunk all articles and build a metadata index
all_chunks = []       # list of text strings
chunk_metadata = []   # parallel list: which article each chunk came from

for article_id, article in articles.items():
    chunks = chunk_text(article['text'], chunk_size=500, overlap=100)
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        chunk_metadata.append({
            'article_id': article_id,
            'title': article['title'],
            'year': article['year'],
            'chunk_index': i,
        })

print(f"Created {len(all_chunks)} chunks from {len(articles)} articles")
print(f"Average chunk length: {np.mean([len(c.split()) for c in all_chunks]):.0f} words")

In [ ]:
import faiss

# Embed all chunks (this takes 30-60 seconds)
print("Embedding all chunks...")
chunk_embeddings = embedder.encode(all_chunks, show_progress_bar=True, batch_size=64)
print(f"Embeddings shape: {chunk_embeddings.shape}")  # (n_chunks, 384)

# Build FAISS index
# We use IndexFlatIP (Inner Product) after normalizing vectors,
# which is equivalent to cosine similarity search.
faiss.normalize_L2(chunk_embeddings)  # normalize in-place so inner product = cosine sim
index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)
print(f"FAISS index built with {index.ntotal} vectors")

In [ ]:
def retrieve(question, k=5):
    """Find the k most relevant chunks for a question.

    Embeds the question, searches the FAISS index, and returns
    the top-k chunks with their metadata and similarity scores.

    Args:
        question: The research question to search for.
        k: Number of chunks to retrieve.

    Returns:
        List of dicts with 'text', 'title', 'article_id', 'score'.
    """
    q_vec = embedder.encode([question])
    faiss.normalize_L2(q_vec)
    scores, indices = index.search(q_vec, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            'text': all_chunks[idx],
            'score': float(score),
            **chunk_metadata[idx],
        })
    return results


# Test retrieval
test_results = retrieve("How do states regulate artificial intelligence?", k=3)
for i, r in enumerate(test_results):
    print(f"\n--- Result {i+1} (score: {r['score']:.4f}) ---")
    print(f"From: {r['title'][:60]} ({r.get('year', '')})")
    print(r['text'][:300] + "...")

## Exercise 5: Build the RAG Generation Step

You have retrieval working. Now you need the generation step: given a question and retrieved passages, write a prompt that produces a grounded answer.

The key constraint: **the model should only use information from the provided passages**. Every claim in the answer should be traceable to a specific passage. This is where the provenance thread from Section 1 continues.

You'll try this with both the local model (3B) and the API model (235B) to see how scale affects generation quality.

In [ ]:
# @title Exercise 5: Write a RAG generation prompt
# Your prompt should make the model:
# 1. Only use information from the passages
# 2. Cite which passage each claim comes from
# 3. Say "I don't have enough information" if passages don't answer the question
#
# Hints: number the passages, specify citation format, tell it not to use prior knowledge

def rag_answer(question, k=5, use_api=False):
    """Answer a question using RAG: retrieve relevant passages, then generate.

    Args:
        question: The research question.
        k: Number of passages to retrieve.
        use_api: If True, use the 235B API model; otherwise use local 3B model.

    Returns:
        Tuple of (answer_text, retrieved_passages).
    """
    # Step 1: Retrieve
    retrieved = retrieve(question, k=k)

    # Step 2: Build the prompt with retrieved context
    context = ""
    for i, r in enumerate(retrieved):
        context += f"\n[Passage {i+1}] (from: {r['title'][:60]}, {r.get('year', '')})\n{r['text']}\n"

    prompt = f"""
# TODO: Write your RAG prompt here.
# Use the `context` variable (which contains the retrieved passages)
# and the `question` variable.
# Remember: the model should ONLY use information from the passages.

{context}

Question: {question}
"""

    # Step 3: Generate
    if use_api:
        answer = generate_api(prompt, max_tokens=1024, temperature=0.1)
    else:
        answer = generate(prompt, max_new_tokens=1024, temperature=0.1)

    return answer, retrieved


# Run RAG with the local model
question = "How do states approach the governance and regulation of artificial intelligence?"
print("Answering with LOCAL model (Qwen2.5-3B)...\n")
answer_local, sources = rag_answer(question, use_api=False)
print(answer_local)

In [ ]:
# Now the same question with the API model
print("Answering with API model (Qwen3-235B)...\n")
answer_api, _ = rag_answer(question, use_api=True)
print(answer_api)

In [ ]:
# Compare side by side
print("=" * 80)
print("LOCAL MODEL (3B parameters)")
print("=" * 80)
print(answer_local[:800])
print()
print("=" * 80)
print("API MODEL (235B parameters)")
print("=" * 80)
print(answer_api[:800])
print()
print("💡 Compare: Which answer is more grounded in the sources?")
print("   Which one adds information not in the passages?")
print("   Which one cites sources more reliably?")

## Exercise 6: Evaluating RAG — Grounding vs. Hallucination

The whole point of RAG is to ground the model's answers in real sources. Does it actually work? You'll test this by comparing:

1. **RAG answer**: The model sees retrieved passages and answers based on them
2. **No-context answer**: The model answers from its own knowledge (no passages)

If RAG is working, the RAG answer should be more specific, more accurate, and traceable to sources. The no-context answer might sound plausible but include fabricated details.

You'll also check whether the RAG answer actually uses the retrieved passages or ignores them.

In [ ]:
# @title Exercise 6: Compare RAG vs. no-context generation

# Step 1: No-context answer (the model uses only its training data)
no_context_prompt = f"""Answer the following research question based on your knowledge
of political science. Be specific and cite any relevant scholarship you know about.

Question: {question}"""

print("NO-CONTEXT answer (API model, no retrieved passages):\n")
no_context_answer = generate_api(no_context_prompt, max_tokens=1024)
print(no_context_answer)

In [ ]:
# Step 2: Provenance check — does the RAG answer actually use the sources?

def check_provenance(answer, retrieved_passages):
    """For each sentence in the answer, find the most similar retrieved passage.
    Shows the matching snippet so you can judge whether it's real grounding
    or just topical similarity.
    """
    sentences = [s.strip() for s in answer.split('.') if len(s.strip()) > 20]
    passage_texts = [r['text'] for r in retrieved_passages]

    # Embed sentences and passages
    sent_vecs = embedder.encode(sentences)
    passage_vecs = embedder.encode(passage_texts)

    results = []
    for i, sent in enumerate(sentences):
        sims = [cosine_similarity(sent_vecs[i], pv) for pv in passage_vecs]
        best_idx = int(np.argmax(sims))
        best_score = sims[best_idx]
        results.append({
            'sentence': sent,
            'best_passage': best_idx + 1,
            'similarity': best_score,
            'grounded': best_score > 0.4,
            'matching_snippet': passage_texts[best_idx][:150],  # show what it matched
        })

    return results


# Run on RAG answer
provenance_rag = check_provenance(answer_api, sources)

grounded_count = sum(1 for p in provenance_rag if p['grounded'])
print(f"=== PROVENANCE CHECK: RAG ANSWER ===")
print(f"{grounded_count}/{len(provenance_rag)} sentences appear grounded in retrieved passages\n")

for p in provenance_rag:
    status = "✓ GROUNDED" if p['grounded'] else "⚠ UNGROUNDED"
    print(f"{status} (passage {p['best_passage']}, sim={p['similarity']:.3f})")
    print(f"  Answer:  \"{p['sentence'][:90]}...\"")
    print(f"  Matched: \"{p['matching_snippet'][:90]}...\"")
    print()

# Now run the SAME check on the no-context answer
provenance_no_ctx = check_provenance(no_context_answer, sources)

grounded_no_ctx = sum(1 for p in provenance_no_ctx if p['grounded'])
print(f"\n=== PROVENANCE CHECK: NO-CONTEXT ANSWER ===")
print(f"{grounded_no_ctx}/{len(provenance_no_ctx)} sentences appear grounded in retrieved passages\n")

for p in provenance_no_ctx:
    status = "✓ GROUNDED" if p['grounded'] else "⚠ UNGROUNDED"
    print(f"{status} (passage {p['best_passage']}, sim={p['similarity']:.3f})")
    print(f"  Answer:  \"{p['sentence'][:90]}...\"")
    print(f"  Matched: \"{p['matching_snippet'][:90]}...\"")
    print()

print("💡 Compare: does the no-context answer score surprisingly well?")
print("   Read the matched snippets — is the checker measuring real grounding, or just topical similarity?")

### Reflection

Compare the three outputs:
- **RAG + local model (3B)**: likely struggles with synthesis but might stick close to the text
- **RAG + API model (235B)**: better synthesis, but does it stay grounded?
- **No-context answer**: plausible but potentially fabricated

**A critical question about the provenance checker**: you may notice that the no-context answer scores nearly as well as the RAG answer. Does that mean the model "knew" the right answer already? Probably not. The checker uses **cosine similarity**, which measures topical overlap — a hallucinated sentence about AI governance will be similar to real passages about AI governance simply because they share vocabulary and concepts. High similarity ≠ grounding. This is a fundamental limitation of embedding-based evaluation, and it's why social scientists can't rely on automated metrics alone. You need to read the matched snippets and judge for yourself.

> **Stretch goal**: Try a question that spans multiple topics in the corpus — e.g., *"How do corruption, sanctions, and democratic backsliding interact?"* This requires the model to retrieve and synthesize across articles. Does RAG handle cross-topic questions well? Try different values of `k` (number of retrieved passages) and see how it affects the answer.

> **Stretch goal 2**: Experiment with chunk size. Re-chunk the corpus with `chunk_size=200` and `chunk_size=1000`. How does this affect retrieval quality? Smaller chunks are more precise but lose context; larger chunks keep context but may dilute relevance.

---

# Bridge to Day 5: From Pipeline to Agent

Look at what you've built today:

- `generate()` / `generate_api()` — functions that call a language model
- `retrieve()` — a function that searches a corpus for relevant passages
- `rag_answer()` — a function that combines retrieval and generation
- `verify_faithfulness()` — a function that checks whether claims are grounded

Each of these is a **tool** — a function with a clear input and output. Tomorrow, you'll connect these tools to an **agent**: a system that can *decide* which tool to use and when.

Instead of you writing `retrieve(question)` → `generate(prompt)`, an agent could:

1. Receive a research question
2. *Decide* it needs to search the corpus → call `retrieve()`
3. Read the results and *decide* it needs more context → call `retrieve()` again with a refined query
4. Synthesize an answer → call `generate()`
5. *Decide* to check its own work → call `verify_faithfulness()`

The pipeline becomes a **planning loop**. That's Day 5.

---

### Summary: What you learned today

| Concept | What it means | Why it matters for research |
|---|---|---|
| **Information extraction** | Getting structured data from unstructured text | Automates literature coding at scale |
| **Faithfulness verification** | Checking that extracted info is actually in the source | Prevents data contamination |
| **Chunking** | Splitting documents into pieces for retrieval | Determines what your system can find |
| **Embedding-based retrieval** | Finding relevant passages by vector similarity | Scales to large corpora without keywords |
| **RAG** | Combining retrieval with generation | Grounds model answers in real sources |
| **Provenance** | Tracing every claim back to its source | The methodological requirement for trust |

### Further reading

- Lewis, P. et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* — The foundational RAG paper
- Gao, Y. et al. (2024). *Retrieval-Augmented Generation for Large Language Models: A Survey.* — Comprehensive RAG overview
- Anthropic (2024). [Contextual Retrieval](https://www.anthropic.com/news/contextual-retrieval) — Improved chunking strategies for RAG

---

# Solutions

> **Don't look at these until you've attempted the exercises!** Struggling with the exercise is where the learning happens. These are reference implementations, not the only correct answers.

In [ ]:
# @title Solution: Exercise 1 — Manual extraction (example)
# There's no single correct answer here, but a good extraction might look like:

manual_extraction_solution = {
    "claims": [
        "AI governance differs from previous governance challenges because no single human fully understands how generative AI arrives at its responses.",
        "Existing governance structures were not designed to contain the pervasive risks posed by AI.",
        "The failures of current internet and data governance have highlighted trade-offs that AI governance must learn from.",
    ],
    "cited_findings": [
        {"finding": "RLHF can be applied as a band-aid to prevent unwanted model behavior", "source": "described in context of Kevin Roose incident"},
        {"finding": "The velocity of AI development creates governance challenges comparable to historical moments of profound technological change", "source": "Ord 2022"},
    ],
    "methods_or_frameworks": [
        "The authors use a comparative framework across open and closed societies to analyze governance approaches.",
        "They distinguish between AI safety (technical) and AI governance (institutional) as related but distinct concepts.",
    ],
}

import json
print(json.dumps(manual_extraction_solution, indent=2))

In [ ]:
# @title Solution: Exercise 2 — Extraction prompt

extraction_prompt_solution = f"""Extract structured information from the following academic passage.
Return a JSON object with three keys:

1. "claims": A list of substantive assertions the authors make. Each claim should be a dict with:
   - "claim": the assertion (paraphrased)
   - "source_quote": a short quote from the passage that supports this claim

2. "cited_findings": A list of findings attributed to other researchers. Each should be a dict with:
   - "finding": what was found
   - "citation": the author(s) and year as given in the text
   - "source_quote": the relevant quote from the passage

3. "methods_or_frameworks": A list of analytical approaches or frameworks mentioned. Each should be a string.

IMPORTANT RULES:
- Only extract information that is explicitly stated in the passage.
- Do NOT add information from your own knowledge.
- Every claim must have a source_quote that appears in the passage.
- If you are unsure whether something is a claim or a cited finding, include it in both.
- Return ONLY valid JSON, no other text.

Passage:
{passage}
"""

llm_response_sol = generate(extraction_prompt_solution, max_new_tokens=1500, temperature=0.1)
llm_extraction_sol = parse_json_response(llm_response_sol)
if llm_extraction_sol:
    print(json.dumps(llm_extraction_sol, indent=2))
else:
    print("Parsing failed. Raw response:")
    print(llm_response_sol[:500])

In [ ]:
# @title Solution: Exercise 5 — RAG generation prompt

def rag_answer_solution(question, k=5, use_api=False):
    """RAG with a well-structured generation prompt."""
    retrieved = retrieve(question, k=k)

    context = ""
    for i, r in enumerate(retrieved):
        context += f"\n[Passage {i+1}] (from: {r['title'][:60]}, {r.get('year', '')})\n{r['text']}\n"

    prompt = f"""You are a research assistant answering questions about political science.
You have been given {len(retrieved)} passages retrieved from academic review articles.

INSTRUCTIONS:
- Answer the question using ONLY information from the passages below.
- For each claim in your answer, cite the passage number in brackets, e.g. [Passage 2].
- If the passages don't contain enough information to answer the question, say so explicitly.
- Do NOT use any knowledge beyond what is in the passages.
- Be concise but thorough.

PASSAGES:
{context}

QUESTION: {question}

ANSWER:"""

    if use_api:
        answer = generate_api(prompt, max_tokens=1024, temperature=0.1)
    else:
        answer = generate(prompt, max_new_tokens=1024, temperature=0.1)

    return answer, retrieved


question = "How do states approach the governance and regulation of artificial intelligence?"
print("=== RAG answer (API model) ===\n")
answer, sources = rag_answer_solution(question, use_api=True)
print(answer)
print("\n\n=== Sources used ===")
for i, s in enumerate(sources):
    print(f"  [Passage {i+1}] {s['title'][:60]} ({s.get('year', '')}) — similarity: {s['score']:.3f}")